# Child dangerous-object warning system — Module A on Kaggle

ChildSUn safe/unsafe classifier: leak-free **group split** vs **random split**, 3 models
(EfficientNet-B0, MobileNetV3-Large, VGG19-BN), temperature scaling, corruption curves, Grad-CAM.

**Before running**
1. *Add Input* → search **ChildSUn** (`fairuzkhan002/datasetvgg19`) → Add.
2. *Settings* → **Accelerator: GPU T4 x2** (or T4). The P100 may not be supported by the current PyTorch build.
3. *Settings* → **Internet: On** (needed for `pip install imagehash` and the pretrained ImageNet weights).
4. *Run All*. Results go to `/kaggle/working/childsafe/outputs` and `docs/`; the last cell zips them for download.

Module B (webcam + MediaPipe) cannot run on Kaggle — run it locally.

## 1. Setup

In [ ]:
!pip install -q imagehash timm
import os, sys, glob, subprocess
REPO = "/kaggle/working/childsafe"
for d in ["configs", "scripts", "src/childsafe/data", "src/childsafe/classifier", "docs", "outputs", "data/splits"]:
    os.makedirs(f"{REPO}/{d}", exist_ok=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
os.environ["PYTHONPATH"] = f"{REPO}/src"
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

## 2. Project source (generated from the repo — expand to read)

In [ ]:
%%writefile configs/default.yaml
# Central config. Every threshold here is a starting value, NOT a tuned result.
# Tune D3 thresholds by grid search on the dev split of the scenario videos (scripts/grid_search.py),
# and report on the held-out test split only.

paths:
  raw_data: data/raw/childsun          # extracted ChildSUn root
  splits: data/splits/splits.csv
  hashes: data/splits/hashes.npz
  outputs: outputs
  docs: docs                           # generated reports (dataset, split, results)
  face_model: models/face_landmarker.task
  hand_model: models/hand_landmarker.task

# Category name (folder name, or filename prefix for the flat ChildSUn layout; normalized:
# lowercase, no spaces/_/-) -> binary label. Grouping checked against the Kaggle dataset
# description (2026-10-01): unsafe = knives, sockets, scissors, electric heaters, forks;
# safe = toys, baby items, fruits (spoon and ball are not named there; kept safe).
label_map:
  knife: unsafe
  socket: unsafe
  sockets: unsafe
  scissors: unsafe
  scissor: unsafe
  seissor: unsafe     # typo used in ChildSUn filenames
  electricheater: unsafe
  heater: unsafe
  fork: unsafe
  toy: safe
  toys: safe
  ball: safe
  babyitems: safe
  babyitem: safe
  fruit: safe
  fruits: safe
  spoon: safe
  # objects outside the 10 ChildSUn classes (Open Images pen, phone, mug, hand...). "safe" here only means
  # "not an unsafe class" for P(unsafe); it is not a safety claim (a pen cap is a choking hazard).
  other: safe
  # if the dataset is already organised as safe/ and unsafe/ folders:
  safe: safe
  unsafe: unsafe

# Merge spelling variants into one class name.
# socket/sockets are separate filename sets in ChildSUn (86 + 124 source photos); merged here
# because the paper lists a single "socket" category.
category_aliases:
  sockets: socket
  seissor: scissors
  scissor: scissors
  fruits: fruit
  toys: toy
  babyitem: babyitems
  electricheater: heater

dedup:
  hash_size: 8              # 64-bit pHash / dHash
  phash_threshold: 8        # chosen by inspecting montages (docs/split_report.md): true dups have low dHash
  dhash_threshold: 5
  dihedral: true            # also compare flipped / 90-degree-rotated variants (catches flip/rotate augmentation)

split:
  test_frac: 0.15
  val_frac: 0.15
  seed: 42

train:
  img_size: 224
  batch_size: 32
  epochs: 25
  lr: 3.0e-4
  lr_overrides:             # VGG19 diverged at 3e-4 (best epoch 0 in the first runs)
    vgg19_bn: 5.0e-5
  weight_decay: 1.0e-4
  patience: 6
  num_workers: 4
  amp: true
  robust_aug: false         # A5: random erasing, motion blur, gamma, ISO noise
  models: [efficientnet_b0, mobilenetv3_large_100, vgg19_bn]

calibration:
  n_bins: 15
  reject_threshold: 0.70    # calibrated max prob below this -> "uncertain"

landmarks:
  normalizer: face_width    # face_width | inter_eye | hand_size (B1 compares all three)
  min_face_conf: 0.5
  min_hand_conf: 0.5
  num_hands: 2

smoothing:
  method: one_euro          # none | ema | one_euro
  ema_alpha: 0.4
  one_euro_min_cutoff: 1.0
  one_euro_beta: 0.05
  lost_hold_s: 0.8          # B2: hold last state this long when the mouth landmark disappears
  lost_max_s: 3.0           # legacy metadata only; runtime never extends held evidence beyond lost_hold_s

temporal:                   # D1
  window_s: 0.3333333333333333
  required_evidence_s: 0.1
  max_gap_s: 0.25
  d_enter: 0.35             # normalized distance to enter "near" (hysteresis: enter < exit)
  d_exit: 0.50
  n_of_m: [3, 10]           # legacy metadata only; runtime uses the duration window above
  merge_s: 2.0              # merge events closer than this
  cooldown_s: 5.0           # limit repeat notifications; never lower displayed danger

cascade:                    # D2
  enabled: true
  d_far: 1.2                # run Module A only when normalized distance < d_far
  classify_every_n: 3       # and at most every N frames (cached in between)
  classify_interval_s: 0.1
  crop_scale: 2.2           # hand crop = hand bbox * scale

rules:                      # D3
  p_unsafe_danger: 0.60     # calibrated P(unsafe) needed for danger
  dwell_danger_s: 0.5       # near for at least this long before danger
  dwell_safe_caution_s: 3.0 # safe object near mouth this long -> caution (capped, never danger)
  ttc_caution_s: 0.8        # unsafe object approaching with time-to-contact below this -> caution

runtime:
  max_people: 4
  stale_frame_s: 0.25
  camera_index: 0
  width: 1280
  height: 720
  log_dir: outputs/events   # events only; video is never written

detector:
  weights: models/yolo11n-seg.pt
  min_conf: 0.5
  complete_object_coverage: false # COCO excludes coins/pens; never infer their location

quality:
  calibrated: false
  max_hand_object_px: 80
  min_face_px: 40
  min_hand_px: 10
  min_mouth_px: 4
  max_yaw_deg: 60
  max_pitch_deg: 45
  max_landmark_jump: 0.35
  min_sharpness: 20
  min_brightness: 40
  max_brightness: 220
  min_mask_px: 16
  min_mask_conf: 0.6

In [ ]:
%%writefile src/childsafe/__init__.py
"""Child dangerous-object warning system."""

In [ ]:
%%writefile src/childsafe/config.py
"""Config loading. Paths in the YAML are relative to the repo root.

A config file may start with `_base_: default.yaml` and list only the keys it changes
(e.g. configs/tuned_overrides.yaml written by the grid search).
"""
from __future__ import annotations

from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parents[2]
DEFAULT_CONFIG = ROOT / "configs" / "default.yaml"


def _merge(base: dict, over: dict) -> dict:
    out = dict(base)
    for k, v in over.items():
        out[k] = _merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


def _read(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    base = cfg.pop("_base_", None)
    return _merge(_read(path.parent / base), cfg) if base else cfg


def load_config(path: str | Path | None = None) -> dict:
    cfg = _read(Path(path) if path else DEFAULT_CONFIG)
    cfg["paths"] = {k: str((ROOT / v).resolve()) for k, v in cfg["paths"].items()}
    if cfg.get('detector',{}).get('weights'):
        cfg['detector']['weights'] = str((ROOT / cfg['detector']['weights']).resolve())
    return cfg


def normalize_name(name: str) -> str:
    return "".join(ch for ch in name.lower() if ch.isalnum())

In [ ]:
%%writefile src/childsafe/data/__init__.py
# package

In [ ]:
%%writefile src/childsafe/data/catalog.py
"""Build an image catalog (path, category, binary label) from a folder tree.

The ChildSUn folder layout was not verified before writing this, so the category is
inferred from folder names: the deepest folder whose normalized name is in
``label_map`` wins, and a specific category (``knife``) beats a generic one (``unsafe``).
Folders called train/test/val are kept as ``source_split`` for information only;
they are NOT used for our split because they may contain leaked augmentations.
"""
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

from childsafe.config import normalize_name

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_NAMES = {"train", "training", "test", "testing", "val", "valid", "validation"}
GENERIC = {"safe", "unsafe"}


def infer_labels(rel_parts: tuple[str, ...], label_map: dict) -> tuple[str | None, str | None, str | None]:
    """Return (category, binary_label, source_split) from the folder parts of a relative path."""
    category = binary = generic = source_split = None
    for part in rel_parts:
        norm = normalize_name(part)
        if norm in SPLIT_NAMES:
            source_split = norm
        if norm in label_map:
            if norm in GENERIC:
                generic = norm
            else:
                category, binary = norm, label_map[norm]
    if category is None and generic is not None:
        category, binary = generic, generic
    return category, binary, source_split


AUG_NAME = re.compile(r"^(?P<prefix>.+?)_(?P<src>\d+)_\d+_\d+$")   # Keras save_to_dir: <prefix>_<index>_<n>_<rand>
PLAIN_NAME = re.compile(r"^(?P<prefix>.+?)_\d+$")                   # e.g. fruits_12


def parse_filename(stem: str) -> tuple[str | None, str | None]:
    """Return (class prefix, source photo index or None)."""
    m = AUG_NAME.match(stem)
    if m:
        return m["prefix"], m["src"]
    m = PLAIN_NAME.match(stem)
    return (m["prefix"], None) if m else (None, None)


def source_roots(cfg: dict) -> dict[str, Path]:
    """Image root per source: ChildSUn plus any `extra_sources` (name -> folder relative to the repo)."""
    from childsafe.config import ROOT
    roots = {"childsun": Path(cfg["paths"]["raw_data"])}
    roots.update({name: (ROOT / folder).resolve() for name, folder in (cfg.get("extra_sources") or {}).items()})
    return roots


def build_full_catalog(cfg: dict) -> pd.DataFrame:
    """ChildSUn catalog plus extra sources (e.g. Open Images crops), with a `source` column.
    Extra-source crops are grouped by their original photo id (file name `<imageid>_<class>.jpg`)."""
    frames = []
    for name, root in source_roots(cfg).items():
        df = build_catalog(root, cfg["label_map"], cfg.get("category_aliases"))
        df["source"] = name
        if name != "childsun":
            df["source_id"] = name + "_" + df["path"].map(lambda p: Path(p).stem.split("_")[0])
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def build_catalog(root: str | Path, label_map: dict, aliases: dict | None = None) -> pd.DataFrame:
    root = Path(root)
    aliases = aliases or {}
    rows, unmatched = [], {}
    # sort by the posix string: Path sorting is case-insensitive on Windows, case-sensitive on Linux,
    # which gave different image orders (and different group splits) locally and on Kaggle
    for p in sorted(root.rglob("*"), key=lambda q: q.relative_to(root).as_posix()):
        if p.suffix.lower() not in IMG_EXT or not p.is_file():
            continue
        rel = p.relative_to(root)
        category, binary, source_split = infer_labels(rel.parts[:-1], label_map)
        prefix, source = parse_filename(p.stem)
        if category is None or category in GENERIC:
            # flat layout (as ChildSUn ships): class is the filename prefix
            norm = normalize_name(prefix) if prefix else ""
            if norm in label_map and norm not in GENERIC:
                category, binary = norm, label_map[norm]
        if category is None:
            key = f"{rel.parent} ({prefix or p.stem})"
            unmatched[key] = unmatched.get(key, 0) + 1
            continue
        category = aliases.get(category, category)
        rows.append({"path": rel.as_posix(), "category": category, "label": binary,
                     "source_split": source_split or "",
                     # augmented copies of one original photo share this id
                     "source_id": f"{normalize_name(prefix)}_{source}" if source is not None else rel.as_posix()})
    if unmatched:
        print(f"[catalog] {sum(unmatched.values())} images not matched to label_map (skipped):")
        for k, v in sorted(unmatched.items()):
            print(f"    {k}: {v}")
    if not rows:
        raise SystemExit(f"No labelled images found under {root}. Check paths.raw_data and label_map.")
    return pd.DataFrame(rows)

In [ ]:
%%writefile src/childsafe/data/dedup.py
"""A1: perceptual-hash near-duplicate grouping.

Each image gets a 64-bit pHash and dHash. With ``dihedral=True`` we also hash the 7
flipped/rotated variants, and the distance between images i and j is the minimum over
j's variants. That catches copies produced by flip / 90-degree rotation augmentation,
which a plain pHash misses. Small-angle rotations, crops and colour jitter mostly keep
pHash close anyway. Two images are near-duplicates when BOTH hash distances are under
threshold; near-duplicate pairs are merged into groups with union-find.

Hash thresholds are a judgement call: inspect the montage from scripts/find_duplicates.py
at the chosen threshold and report the threshold in the write-up.
"""
from __future__ import annotations

from pathlib import Path

import imagehash
import numpy as np
from PIL import Image
from tqdm import tqdm

N_VARIANTS = 8


def _dihedral(img: Image.Image) -> list[Image.Image]:
    out = []
    for flip in (False, True):
        base = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT) if flip else img
        for k in range(4):
            out.append(base.rotate(90 * k, expand=True) if k else base)
    return out


def _to_int(h: imagehash.ImageHash) -> np.uint64:
    bits = np.packbits(h.hash.flatten().astype(np.uint8))
    return np.frombuffer(bits.tobytes(), dtype=">u8")[0].astype(np.uint64)


def compute_hashes(paths: list[str | Path], hash_size: int = 8, dihedral: bool = True):
    """Return (phash, dhash) arrays of shape (N, 8) uint64. Unreadable images get all-ones hashes."""
    if hash_size != 8:
        raise ValueError("only 64-bit hashes (hash_size=8) are supported")
    n = len(paths)
    ph = np.full((n, N_VARIANTS), np.iinfo(np.uint64).max, dtype=np.uint64)
    dh = ph.copy()
    bad = []
    for i, p in enumerate(tqdm(paths, desc="hashing")):
        try:
            img = Image.open(p).convert("L")
        except Exception:  # corrupt file: keep sentinel hash so it matches nothing
            bad.append(str(p))
            continue
        variants = _dihedral(img) if dihedral else [img] * N_VARIANTS
        for v, im in enumerate(variants):
            ph[i, v] = _to_int(imagehash.phash(im, hash_size))
            dh[i, v] = _to_int(imagehash.dhash(im, hash_size))
    if bad:
        print(f"[dedup] {len(bad)} unreadable images, e.g. {bad[:3]}")
    return ph, dh


def _popcount(x: np.ndarray) -> np.ndarray:
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x)
    b = x.view(np.uint8).reshape(*x.shape, 8)
    return np.unpackbits(b, axis=-1).sum(-1)


def min_distance_matrix_block(h: np.ndarray, rows: slice, cols: np.ndarray | None = None) -> np.ndarray:
    """Min Hamming distance between canonical hashes h[rows, 0] and all variants of h[cols]."""
    a = h[rows, 0][:, None, None]
    b = h[cols] if cols is not None else h
    return _popcount(a ^ b[None, :, :]).min(-1)


def find_duplicate_pairs(ph, dh, phash_thr: int, dhash_thr: int, chunk: int = 256):
    """Return array of (i, j, d_phash, d_dhash) for i < j that are near-duplicates."""
    n = len(ph)
    out = []
    for s in tqdm(range(0, n, chunk), desc="pairwise"):
        rows = slice(s, min(s + chunk, n))
        dp = min_distance_matrix_block(ph, rows)
        dd = min_distance_matrix_block(dh, rows)
        ii, jj = np.nonzero((dp <= phash_thr) & (dd <= dhash_thr))
        ii = ii + s
        keep = ii != jj
        # the variant-min distance is not symmetric, so keep a match found in either direction
        for i, j in zip(ii[keep], jj[keep]):
            out.append((min(i, j), max(i, j), dp[i - s, j], dd[i - s, j]))
    if not out:
        return np.zeros((0, 4), dtype=np.int64)
    arr = np.array(out, dtype=np.int64)
    arr = arr[np.lexsort((arr[:, 2] + arr[:, 3], arr[:, 1], arr[:, 0]))]
    _, first = np.unique(arr[:, :2], axis=0, return_index=True)
    return arr[np.sort(first)]


def union_find_groups(n: int, pairs: np.ndarray) -> np.ndarray:
    parent = np.arange(n)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i, j in pairs[:, :2]:
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[max(ri, rj)] = min(ri, rj)
    roots = np.array([find(i) for i in range(n)])
    _, group = np.unique(roots, return_inverse=True)
    return group


def cross_split_leakage(ph, dh, split: np.ndarray, phash_thr: int, dhash_thr: int,
                        query: str = "test", ref: str = "train") -> float:
    """Fraction of `query` images that have a near-duplicate in `ref`."""
    q = np.nonzero(split == query)[0]
    r = np.nonzero(split == ref)[0]
    if len(q) == 0 or len(r) == 0:
        return float("nan")
    hits = 0
    for s in range(0, len(q), 256):
        idx = q[s:s + 256]
        dp = _popcount(ph[idx, 0][:, None, None] ^ ph[r][None]).min(-1)
        dd = _popcount(dh[idx, 0][:, None, None] ^ dh[r][None]).min(-1)
        hits += int(((dp <= phash_thr) & (dd <= dhash_thr)).any(1).sum())
    return hits / len(q)

In [ ]:
%%writefile src/childsafe/data/split.py
"""A1: group-based split vs plain random split.

Both splits use the same train/val/test fractions and stratify by category, so the
only difference is whether near-duplicate groups are kept together. The gap in test
accuracy between the two is our estimate of how much leakage inflates the score.

The group split uses its own greedy assignment with numpy's PCG64 generator instead of
sklearn's StratifiedGroupKFold: the sklearn result changed between library versions
(local vs Kaggle gave different splits from identical groups), which breaks reproducibility.
"""
from __future__ import annotations

import numpy as np
from sklearn.model_selection import train_test_split

PARTS = ("train", "val", "test")


def _strata(y: np.ndarray, min_count: int = 3) -> np.ndarray:
    """Stratification labels; classes too small to stratify are pooled (sklearn needs >= 2 per class)."""
    y = np.asarray(y, dtype=object)
    vals, counts = np.unique(y, return_counts=True)
    rare = set(vals[counts < min_count])
    if not rare:
        return y
    s = np.array(["__rare__" if v in rare else v for v in y], dtype=object)
    if (s == "__rare__").sum() < min_count:
        s[s == "__rare__"] = vals[counts.argmax()]
    return s


def random_split(y: np.ndarray, test_frac: float, val_frac: float, seed: int) -> np.ndarray:
    idx = np.arange(len(y))
    trval, test = train_test_split(idx, test_size=test_frac, stratify=_strata(y), random_state=seed)
    train, val = train_test_split(trval, test_size=val_frac / (1 - test_frac),
                                  stratify=_strata(np.asarray(y)[trval]), random_state=seed)
    out = np.empty(len(y), dtype=object)
    out[train], out[val], out[test] = "train", "val", "test"
    return out


def group_split(y: np.ndarray, groups: np.ndarray, test_frac: float, val_frac: float,
                seed: int) -> np.ndarray:
    """Greedy stratified group split: groups are visited largest first (random order within a
    size), and each goes to the part that still needs the most images of the group's classes."""
    y = np.asarray(y)
    groups = np.asarray(groups)
    classes, y_idx = np.unique(y, return_inverse=True)
    gids, g_idx = np.unique(groups, return_inverse=True)
    counts = np.zeros((len(gids), len(classes)), dtype=np.int64)
    np.add.at(counts, (g_idx, y_idx), 1)

    frac = np.array([1 - test_frac - val_frac, val_frac, test_frac])
    target = frac[:, None] * counts.sum(0)[None, :]          # (3, n_classes)
    have = np.zeros_like(target)
    rng = np.random.Generator(np.random.PCG64(seed))
    order = rng.permutation(len(gids))
    order = order[np.argsort(-counts[order].sum(1), kind="stable")]

    assign = np.empty(len(gids), dtype=np.int64)
    for g in order:
        need = ((target - have) * counts[g][None, :]).sum(1) / counts[g].sum()
        part = int(np.argmax(need))
        assign[g] = part
        have[part] += counts[g]

    out = np.array(PARTS, dtype=object)[assign[g_idx]]
    for a in ("val", "test"):
        assert not (set(groups[out == "train"]) & set(groups[out == a])), f"group leaked into {a}"
    return out

In [ ]:
%%writefile src/childsafe/data/corruptions.py
"""A5: image corruptions with a 1-5 severity scale.

Used two ways: (1) as random training augmentation (``RobustAug``), (2) at fixed
severity on the test set to draw degradation curves (accuracy vs severity).
Severity 0 means no corruption. The parameter tables are our own choice, loosely
following the ImageNet-C idea; they are not taken from that benchmark.
"""
from __future__ import annotations

import random

import cv2
import numpy as np
from PIL import Image

SEVERITY = {
    "gaussian_noise": [0.04, 0.08, 0.12, 0.18, 0.26],   # std, image in [0,1]
    "iso_noise":      [0.04, 0.08, 0.12, 0.18, 0.26],   # shot + colour noise
    "motion_blur":    [5, 9, 13, 19, 27],               # kernel length (px at 224)
    "low_light":      [1.5, 2.0, 2.6, 3.3, 4.2],        # gamma > 1 darkens
    "occlusion":      [0.08, 0.15, 0.22, 0.30, 0.40],   # erased area fraction
    "jpeg":           [60, 40, 25, 15, 8],              # quality
}
KINDS = list(SEVERITY)


def _rng(seed):
    return np.random.default_rng(seed)


def corrupt(img: np.ndarray, kind: str, severity: int, seed: int | None = None) -> np.ndarray:
    """img: HxWx3 uint8 RGB. Returns uint8 RGB."""
    if severity <= 0:
        return img
    p = SEVERITY[kind][severity - 1]
    rng = _rng(seed)
    x = img.astype(np.float32) / 255.0
    if kind == "gaussian_noise":
        x = x + rng.normal(0, p, x.shape)
    elif kind == "iso_noise":
        s = 1.0 / p ** 2                      # shot-noise std is p at full brightness
        x = rng.poisson(x * s) / s + rng.normal(0, p / 2, x.shape)
    elif kind == "motion_blur":
        k = int(p)
        kernel = np.zeros((k, k), np.float32)
        kernel[k // 2, :] = 1.0 / k
        angle = rng.uniform(0, 180)
        m = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), angle, 1.0)
        kernel = cv2.warpAffine(kernel, m, (k, k))
        kernel /= max(kernel.sum(), 1e-6)
        x = cv2.filter2D(x, -1, kernel)
    elif kind == "low_light":
        x = np.power(x, p)
        x = x + rng.normal(0, 0.01 * p, x.shape)
    elif kind == "occlusion":
        h, w = x.shape[:2]
        area = p * h * w
        ar = rng.uniform(0.5, 2.0)
        eh, ew = int(min(h, np.sqrt(area * ar))), int(min(w, np.sqrt(area / ar)))
        y0, x0 = rng.integers(0, h - eh + 1), rng.integers(0, w - ew + 1)
        x[y0:y0 + eh, x0:x0 + ew] = rng.uniform(0, 1, 3)
    elif kind == "jpeg":
        ok, buf = cv2.imencode(".jpg", (np.clip(x, 0, 1) * 255).astype(np.uint8)[..., ::-1],
                               [cv2.IMWRITE_JPEG_QUALITY, int(p)])
        return cv2.imdecode(buf, cv2.IMREAD_COLOR)[..., ::-1].copy()
    else:
        raise ValueError(kind)
    return (np.clip(x, 0, 1) * 255).astype(np.uint8)


class FixedCorruption:
    """PIL -> PIL transform with a fixed kind/severity. The seed advances per call, so each image gets
    different noise / occlusion placement, and a fresh instance reproduces the same sequence."""

    def __init__(self, kind: str, severity: int, seed: int = 0):
        self.kind, self.severity, self.seed = kind, severity, seed

    def __call__(self, img: Image.Image) -> Image.Image:
        self.seed += 1
        return Image.fromarray(corrupt(np.asarray(img.convert("RGB")), self.kind, self.severity, self.seed))


class RobustAug:
    """Training-time A5 augmentation: with prob p apply one random corruption at severity 1-3."""

    KINDS = ("iso_noise", "motion_blur", "low_light", "gaussian_noise", "jpeg")

    def __init__(self, p: float = 0.5, max_severity: int = 3):
        self.p, self.max_severity = p, max_severity

    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() > self.p:
            return img
        kind = random.choice(self.KINDS)
        sev = random.randint(1, self.max_severity)
        return Image.fromarray(corrupt(np.asarray(img.convert("RGB")), kind, sev, random.randrange(1 << 30)))

In [ ]:
%%writefile src/childsafe/data/dataset.py
"""Torch dataset and transforms for Module A."""
from __future__ import annotations

from pathlib import Path

import pandas as pd
import torch
from PIL import Image, ImageOps
from torch.utils.data import Dataset
from torchvision import transforms as T

from childsafe.data.corruptions import FixedCorruption, RobustAug

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


class Letterbox:
    """Keep the full thin object visible rather than center-cropping its endpoints."""
    def __init__(self, size):
        self.size = size

    def __call__(self, image):
        return ImageOps.pad(image, (self.size, self.size), method=Image.Resampling.BILINEAR,
                            color=(124, 116, 104))


def train_transform(size: int, robust: bool = False, preprocessing: str = "center_crop"):
    if preprocessing not in ('center_crop','letterbox'):
        raise ValueError('unsupported checkpoint preprocessing')
    tf = [Letterbox(size) if preprocessing == "letterbox" else T.RandomResizedCrop(size, scale=(0.6, 1.0)),
          T.RandomHorizontalFlip(),
          T.ColorJitter(0.3, 0.3, 0.2, 0.02)]
    if robust:
        tf.append(RobustAug(p=0.5))
    tf += [T.ToTensor(), T.Normalize(MEAN, STD)]
    if robust:
        tf.append(T.RandomErasing(p=0.4, scale=(0.02, 0.25), value="random"))
    return T.Compose(tf)


def eval_transform(size: int, corruption: tuple[str, int] | None = None, preprocessing: str = "center_crop"):
    if preprocessing not in ('center_crop','letterbox'):
        raise ValueError('unsupported checkpoint preprocessing')
    tf = [Letterbox(size)] if preprocessing == "letterbox" else [T.Resize(int(size * 1.14)), T.CenterCrop(size)]
    if corruption and corruption[1] > 0:
        tf.append(FixedCorruption(*corruption))
    tf += [T.ToTensor(), T.Normalize(MEAN, STD)]
    return T.Compose(tf)


def image_paths(df: pd.DataFrame, root: str | Path | dict) -> list[Path]:
    """root: one folder, or {source: folder} when df has a `source` column (ChildSUn + extra sources)."""
    if isinstance(root, dict):
        src = df["source"] if "source" in df else ["childsun"] * len(df)
        return [Path(root.get(s, root["childsun"])) / p for s, p in zip(src, df["path"])]
    return [Path(root) / p for p in df["path"]]


class CatalogDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root: str | Path | dict, classes: list[str], transform):
        self.paths = image_paths(df, root)
        idx = {c: i for i, c in enumerate(classes)}
        self.targets = torch.tensor([idx[c] for c in df["category"]])
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.targets[i]

In [ ]:
%%writefile src/childsafe/classifier/__init__.py
# package

In [ ]:
%%writefile src/childsafe/classifier/models.py
"""Model factory (timm) and checkpoint I/O."""
from __future__ import annotations

from pathlib import Path

import timm
import torch
import torch.nn as nn

ARCHS = ("efficientnet_b0", "mobilenetv3_large_100", "vgg19_bn")


def create_model(arch: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    return timm.create_model(arch, pretrained=pretrained, num_classes=num_classes)


def gradcam_target_layer(model: nn.Module, arch: str) -> nn.Module:
    """Last spatial feature layer. (In timm MobileNetV3 conv_head comes after pooling, so use blocks.)"""
    if arch.startswith(("efficientnet", "mobilenetv3")):
        return model.blocks[-1]
    if arch.startswith("vgg"):
        return model.pre_logits
    raise ValueError(f"no Grad-CAM layer rule for {arch}")


def save_checkpoint(path: str | Path, model: nn.Module, arch: str, classes: list[str],
                    binary_map: dict[str, str], img_size: int, temperature: float = 1.0, **extra):
    torch.save({"arch": arch, "classes": classes, "binary_map": binary_map, "img_size": img_size,
                "temperature": temperature, "state_dict": model.state_dict(), **extra}, path)


def load_checkpoint(path: str | Path, device: str | torch.device = "cpu"):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = create_model(ckpt["arch"], len(ckpt["classes"]), pretrained=False)
    model.load_state_dict(ckpt["state_dict"])
    return model.to(device).eval(), ckpt


def unsafe_mask(classes: list[str], binary_map: dict[str, str]) -> torch.Tensor:
    return torch.tensor([binary_map[c] == "unsafe" for c in classes])

In [ ]:
%%writefile src/childsafe/classifier/calibration.py
"""A3: temperature scaling, ECE, reliability diagrams (Guo et al., ICML 2017 method)."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F


def fit_temperature(logits: np.ndarray, targets: np.ndarray, max_iter: int = 200) -> float:
    """Fit a single temperature T > 0 minimising NLL on held-out (val) logits."""
    lg = torch.as_tensor(logits, dtype=torch.float64)
    y = torch.as_tensor(targets, dtype=torch.long)
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=max_iter, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(lg / log_t.exp(), y)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp().item())


def softmax(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    z = logits / temperature
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def ece(confidence: np.ndarray, correct: np.ndarray, n_bins: int = 15) -> float:
    """Expected calibration error with equal-width bins."""
    edges = np.linspace(0, 1, n_bins + 1)
    total, n = 0.0, len(confidence)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (confidence > lo) & (confidence <= hi)
        if m.any():
            total += m.sum() / n * abs(correct[m].mean() - confidence[m].mean())
    return float(total)


def binary_ece(p_pos: np.ndarray, y_pos: np.ndarray, n_bins: int = 15) -> float:
    """ECE of the binary (unsafe) prediction, using confidence of the predicted side."""
    pred = p_pos >= 0.5
    conf = np.where(pred, p_pos, 1 - p_pos)
    return ece(conf, (pred == y_pos).astype(float), n_bins)


def reliability_diagram(confidence, correct, path: str | Path, n_bins: int = 15, title: str = ""):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    edges = np.linspace(0, 1, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    acc, cnt = np.zeros(n_bins), np.zeros(n_bins)
    for b, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        m = (confidence > lo) & (confidence <= hi)
        cnt[b] = m.sum()
        acc[b] = correct[m].mean() if m.any() else np.nan
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(4.5, 5.5), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
    ax.bar(centers, np.nan_to_num(acc), width=1 / n_bins, edgecolor="black", alpha=0.8)
    ax.set_ylabel("accuracy")
    ax.set_title(f"{title}\nECE = {ece(confidence, correct, n_bins):.3f}")
    ax2.bar(centers, cnt, width=1 / n_bins, color="gray")
    ax2.set_xlabel("confidence")
    ax2.set_ylabel("count")
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)

In [ ]:
%%writefile src/childsafe/classifier/engine.py
"""Inference and metric summaries shared by train.py and evaluate.py."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support)

from childsafe.classifier.calibration import binary_ece, ece, reliability_diagram, softmax


@torch.no_grad()
def run_inference(model, loader, device, amp: bool = True):
    model.eval()
    logits, targets = [], []
    for x, y in loader:
        with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
            out = model(x.to(device, non_blocking=True))
        logits.append(out.float().cpu())
        targets.append(y)
    return torch.cat(logits).numpy(), torch.cat(targets).numpy()


def summarize(logits: np.ndarray, targets: np.ndarray, classes: list[str], binary_map: dict,
              temperature: float = 1.0, reject_threshold: float | None = None,
              n_bins: int = 15, plot_prefix: str | Path | None = None, title: str = "") -> dict:
    """Per-class and binary (unsafe-vs-safe) metrics, before and after temperature scaling."""
    unsafe = np.array([binary_map[c] == "unsafe" for c in classes])
    res: dict = {"n": int(len(targets)), "temperature": temperature}
    present = sorted(set(targets.tolist()))
    for tag, t in (("raw", 1.0), ("calibrated", temperature)):
        p = softmax(logits, t)
        pred = p.argmax(1)
        conf = p.max(1)
        correct = (pred == targets).astype(float)
        p_unsafe = p[:, unsafe].sum(1)
        y_unsafe = unsafe[targets]
        res[f"ece_{tag}"] = ece(conf, correct, n_bins)
        res[f"binary_ece_{tag}"] = binary_ece(p_unsafe, y_unsafe, n_bins)
        res[f"nll_{tag}"] = float(-np.log(np.clip(p[np.arange(len(targets)), targets], 1e-12, 1)).mean())
        if plot_prefix is not None:
            reliability_diagram(conf, correct, f"{plot_prefix}_reliability_{tag}.png", n_bins, f"{title} ({tag})")

    p = softmax(logits, temperature)
    pred = p.argmax(1)
    res["accuracy"] = float(accuracy_score(targets, pred))
    res["macro_f1"] = float(f1_score(targets, pred, labels=present, average="macro", zero_division=0))
    prec, rec, f1, sup = precision_recall_fscore_support(targets, pred, labels=range(len(classes)), zero_division=0)
    res["per_class"] = {c: {"precision": float(prec[i]), "recall": float(rec[i]), "f1": float(f1[i]),
                            "support": int(sup[i])} for i, c in enumerate(classes)}
    res["confusion_matrix"] = confusion_matrix(targets, pred, labels=range(len(classes))).tolist()

    # Binary view: P(unsafe) = sum of unsafe-category probabilities.
    p_unsafe = p[:, unsafe].sum(1)
    y_unsafe = unsafe[targets]
    b_pred = p_unsafe >= 0.5
    tp = int((b_pred & y_unsafe).sum()); fn = int((~b_pred & y_unsafe).sum())
    fp = int((b_pred & ~y_unsafe).sum()); tn = int((~b_pred & ~y_unsafe).sum())
    res["binary"] = {
        "accuracy": (tp + tn) / max(len(targets), 1),
        "unsafe_recall": tp / max(tp + fn, 1),
        "unsafe_precision": tp / max(tp + fp, 1),
        "unsafe_f1": 2 * tp / max(2 * tp + fp + fn, 1),
        "tp": tp, "fn": fn, "fp": fp, "tn": tn,
    }
    if reject_threshold is not None:
        # A3 reject option: low calibrated confidence -> "uncertain" (caution), not a hard label.
        b_conf = np.maximum(p_unsafe, 1 - p_unsafe)
        keep = (b_conf >= reject_threshold) & (p.max(1) >= reject_threshold)
        res["reject"] = {
            "threshold": reject_threshold,
            "coverage": float(keep.mean()),
            "binary_accuracy_on_kept": float((b_pred[keep] == y_unsafe[keep]).mean()) if keep.any() else float("nan"),
            "unsafe_missed_as_safe_confidently": int((~b_pred & y_unsafe & keep).sum()),
            "unsafe_rejected_to_caution": int((y_unsafe & ~keep).sum()),
        }
    return res

In [ ]:
%%writefile src/childsafe/classifier/train.py
"""Train one Module A model on one split, then calibrate on val and evaluate on test.

    python -m childsafe.classifier.train --arch efficientnet_b0 --split group
    python -m childsafe.classifier.train --arch efficientnet_b0 --split random   # leakage comparison
    python -m childsafe.classifier.train --arch efficientnet_b0 --split group --robust  # A5

Outputs go to outputs/runs/<arch>_<split>[_robust]/.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from childsafe.classifier.calibration import fit_temperature
from childsafe.classifier.engine import run_inference, summarize
from childsafe.classifier.models import create_model, save_checkpoint
from childsafe.config import load_config
from childsafe.data.catalog import source_roots
from childsafe.data.dataset import CatalogDataset, eval_transform, train_transform


def seed_everything(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", required=True)
    ap.add_argument("--split", choices=["group", "random"], default="group")
    ap.add_argument("--robust", action="store_true", help="A5 robustness augmentation")
    ap.add_argument("--config", default=None)
    ap.add_argument("--epochs", type=int)
    ap.add_argument("--batch-size", type=int)
    ap.add_argument("--no-pretrained", action="store_true")
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--out", default=None)
    ap.add_argument("--manifest", help="Reviewed expanded manifest with train/val/cal/test in split_group")
    ap.add_argument("--balanced", action="store_true", help="Inverse-square-root training class weights")
    ap.add_argument("--preprocessing", choices=["center_crop", "letterbox"], default="center_crop")
    args = ap.parse_args(argv)
    sys.stdout.reconfigure(line_buffering=True)  # progress visible when redirected to a log

    cfg = load_config(args.config)
    tc = cfg["train"]
    epochs = args.epochs or tc["epochs"]
    bs = args.batch_size or tc["batch_size"]
    seed = cfg["split"]["seed"] if args.seed is None else args.seed
    seed_everything(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = tc["amp"] and device.type == "cuda"

    df = pd.read_csv(args.manifest or cfg["paths"]["splits"])
    col = f"split_{args.split}"
    classes = sorted(df["category"].unique())
    binary_map = dict(df[["category", "label"]].drop_duplicates().values)
    if df.groupby("category")["label"].nunique().max() != 1:
        raise ValueError("Conflicting category risk labels")
    if args.manifest:
        if args.split != "group" or not set(df[col]).issubset({"train", "val", "cal", "test"}):
            raise ValueError("Expanded training requires group split with train/val/cal/test")
        if df.groupby("group")[col].nunique().max() != 1:
            raise ValueError("Source group leakage across partitions")
        for part in ("train", "val", "cal", "test"):
            missing = set(classes) - set(df.loc[df[col] == part, "category"])
            if missing:
                raise ValueError(f"{part} missing classes: {sorted(missing)}; collect more independent sources")
    root = source_roots(cfg)   # ChildSUn + any extra sources (paths may also be absolute, e.g. --manifest)
    size = tc["img_size"]

    def loader(part, tf, shuffle):
        ds = CatalogDataset(df[df[col] == part], root, classes, tf)
        sampler = None
        if shuffle and tc.get("balance") == "sqrt":
            # class-balanced sampling, weight ~ 1/sqrt(class count): rare classes are drawn more often
            # without fully flattening the distribution (one epoch still = len(ds) draws)
            counts = torch.bincount(ds.targets, minlength=len(classes)).float().clamp(min=1)
            sampler = torch.utils.data.WeightedRandomSampler(counts.rsqrt()[ds.targets], len(ds), replacement=True)
        return DataLoader(ds, batch_size=bs, shuffle=shuffle and sampler is None, sampler=sampler,
                          num_workers=tc["num_workers"], pin_memory=device.type == "cuda",
                          persistent_workers=tc["num_workers"] > 0, drop_last=shuffle and len(ds) > bs)

    eval_tf = eval_transform(size, preprocessing=args.preprocessing)
    train_dl = loader("train", train_transform(size, args.robust, args.preprocessing), True)
    val_dl = loader("val", eval_tf, False)
    cal_dl = loader("cal", eval_tf, False) if args.manifest else val_dl
    test_dl = loader("test", eval_tf, False)

    run_name = f"{args.arch}_{args.split}" + ("_robust" if args.robust else "")
    out = Path(args.out or Path(cfg["paths"]["outputs"]) / "runs" / run_name)
    out.mkdir(parents=True, exist_ok=True)
    if args.manifest:
        df.to_csv(out / "training_manifest.csv", index=False)
    provenance = {"arguments": vars(args), "seed": seed, "config": cfg,
                  "manifest_sha256": hashlib.sha256(Path(args.manifest or cfg["paths"]["splits"]).read_bytes()).hexdigest(),
                  "partition_counts": df.groupby(col).size().to_dict()}
    (out / "run_config.json").write_text(json.dumps(provenance, indent=2, default=str), encoding="utf-8")

    lr = tc.get("lr_overrides", {}).get(args.arch, tc["lr"])
    print(f"[{run_name}] lr={lr} epochs={epochs} batch={bs}", flush=True)
    model = create_model(args.arch, len(classes), pretrained=not args.no_pretrained).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=tc["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl),
                                                pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=amp)
    weights = None
    if args.balanced:
        counts = df[df[col] == "train"].category.value_counts().reindex(classes).to_numpy()
        weights = torch.tensor(1 / np.sqrt(counts), dtype=torch.float32, device=device)
        weights /= weights.mean()
    crit = nn.CrossEntropyLoss(weight=weights)

    best_f1, best_epoch, history = -1.0, -1, []
    for epoch in range(epochs):
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item() * len(y); n += len(y)
        vl, vy = run_inference(model, val_dl, device, amp)
        vm = summarize(vl, vy, classes, binary_map)
        history.append({"epoch": epoch, "train_loss": tot / max(n, 1), "val_acc": vm["accuracy"],
                        "val_macro_f1": vm["macro_f1"], "val_unsafe_recall": vm["binary"]["unsafe_recall"],
                        "sec": time.time() - t0})
        print(json.dumps(history[-1]), flush=True)
        if vm["macro_f1"] > best_f1:
            best_f1, best_epoch = vm["macro_f1"], epoch
            save_checkpoint(out / "best.pt", model, args.arch, classes, binary_map, size,
                            preprocessing=args.preprocessing)
        elif epoch - best_epoch >= tc["patience"]:
            print(f"early stop at epoch {epoch} (best {best_epoch})")
            break

    # Reload best, calibrate on val, evaluate on test.
    model.load_state_dict(torch.load(out / "best.pt", weights_only=False)["state_dict"])
    vl, vy = run_inference(model, val_dl, device, amp)
    cl, cy = run_inference(model, cal_dl, device, amp)
    temp = fit_temperature(cl, cy)
    save_checkpoint(out / "best.pt", model, args.arch, classes, binary_map, size, temperature=temp,
                    split=args.split, robust=args.robust, best_epoch=best_epoch,
                    preprocessing=args.preprocessing, balanced=args.balanced,
                    calibration_partition="cal" if args.manifest else "val")
    tl, ty = run_inference(model, test_dl, device, amp)
    np.savez(out / "logits.npz", val_logits=vl, val_targets=vy, test_logits=tl, test_targets=ty,
             cal_logits=cl, cal_targets=cy, classes=np.array(classes))
    rc = cfg["calibration"]
    metrics = {
        "arch": args.arch, "split": args.split, "robust": args.robust, "best_epoch": best_epoch,
        "val": summarize(vl, vy, classes, binary_map, temp, rc["reject_threshold"], rc["n_bins"]),
        "test": summarize(tl, ty, classes, binary_map, temp, rc["reject_threshold"], rc["n_bins"],
                          plot_prefix=out / "test", title=f"{run_name} test"),
        "history": history,
        "preprocessing": args.preprocessing, "balanced": args.balanced,
        "calibration_partition": "cal" if args.manifest else "val",
        "test_by_domain": {str(domain): summarize(tl[mask], ty[mask], classes, binary_map, temp,
                                  rc["reject_threshold"], rc["n_bins"])
                           for domain in df.loc[df[col] == "test", "domain"].unique()
                           for mask in [(df.loc[df[col] == "test", "domain"].to_numpy() == domain)]}
                          if "domain" in df else {},
    }
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    t = metrics["test"]
    print(f"[{run_name}] T={temp:.3f} test acc={t['accuracy']:.4f} macroF1={t['macro_f1']:.4f} "
          f"unsafe recall={t['binary']['unsafe_recall']:.4f} ECE raw={t['ece_raw']:.4f} cal={t['ece_calibrated']:.4f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/childsafe/classifier/evaluate.py
"""Evaluate a trained checkpoint in domain, out of domain (A2) and under corruptions (A5).

    python -m childsafe.classifier.evaluate --run outputs/runs/efficientnet_b0_group --ood data/ood
    python -m childsafe.classifier.evaluate --run outputs/runs/efficientnet_b0_group --corruptions

The OOD folder uses the same class folder names as ChildSUn (e.g. data/ood/knife/*.jpg);
extra sub-folders such as data/ood/low_light/knife/... are fine and reported by `condition`.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms as T

from childsafe.classifier.engine import run_inference, summarize
from childsafe.classifier.models import load_checkpoint
from childsafe.config import load_config, normalize_name
from childsafe.data.catalog import build_catalog, source_roots
from childsafe.data.corruptions import KINDS, FixedCorruption
from childsafe.data.dataset import MEAN, STD, eval_transform, image_paths


_CACHE: dict = {}


def _cached_images(df, root, size, preprocessing="center_crop"):
    """Decode + resize + center-crop once; corruption passes reuse these (no DataLoader worker re-spawn,
    which costs tens of seconds per pass on Windows)."""
    key = (str(root), tuple(df["path"]), tuple(df["source"]) if "source" in df else (), size, preprocessing)
    if key not in _CACHE:
        tf = T.Compose(eval_transform(size, preprocessing=preprocessing).transforms[:-2])
        _CACHE[key] = [tf(Image.open(p).convert("RGB")) for p in image_paths(df, root)]
    return _CACHE[key]


def _eval_df(model, ckpt, df, root, device, cfg, corruption=None, plot_prefix=None, title=""):
    size = ckpt["img_size"]
    imgs = _cached_images(df, root, size, ckpt.get("preprocessing", "center_crop"))
    tail = T.Compose(([FixedCorruption(*corruption)] if corruption and corruption[1] > 0 else [])
                     + [T.ToTensor(), T.Normalize(MEAN, STD)])
    idx = {c: i for i, c in enumerate(ckpt["classes"])}
    x = torch.stack([tail(im) for im in imgs])
    y = torch.tensor([idx[c] for c in df["category"]])
    lg, y = run_inference(model, DataLoader(TensorDataset(x, y), batch_size=64), device)
    rc = cfg["calibration"]
    return summarize(lg, y, ckpt["classes"], ckpt["binary_map"], ckpt["temperature"],
                     rc["reject_threshold"], rc["n_bins"], plot_prefix, title)


def _unknown_report(model, ckpt, df, root, device, cfg) -> dict:
    from childsafe.classifier.calibration import softmax
    size = ckpt["img_size"]
    imgs = _cached_images(df, root, size)
    tail = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])
    x = torch.stack([tail(im) for im in imgs])
    lg, _ = run_inference(model, DataLoader(TensorDataset(x, torch.zeros(len(x), dtype=torch.long)), batch_size=64), device)
    p = softmax(lg, ckpt["temperature"])
    unsafe = np.array([ckpt["binary_map"][c] == "unsafe" for c in ckpt["classes"]])
    pu = p[:, unsafe].sum(1)
    thr = cfg["calibration"]["reject_threshold"]
    status = np.where(np.maximum(pu, 1 - pu) < thr, "uncertain", np.where(pu >= 0.5, "unsafe", "safe"))
    pred = np.array(ckpt["classes"])[p.argmax(1)]
    out = {}
    for c in sorted(df["category"].unique()):
        m = (df["category"] == c).to_numpy()
        vc = pd.Series(pred[m]).value_counts(normalize=True).head(3).round(3).to_dict()
        out[c] = {"n": int(m.sum()), "unsafe": float((status[m] == "unsafe").mean()),
                  "uncertain": float((status[m] == "uncertain").mean()), "safe": float((status[m] == "safe").mean()),
                  "mean_max_prob": float(p[m].max(1).mean()), "top_predicted": vc}
    return out


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True, help="run folder containing best.pt")
    ap.add_argument("--ood", default=None, help="out-of-domain image folder (A2)")
    ap.add_argument("--corruptions", action="store_true", help="A5 degradation curves on the test split")
    ap.add_argument("--config", default=None)
    ap.add_argument("--manifest", default=None, help="Expanded split manifest used to train the checkpoint")
    args = ap.parse_args(argv)

    cfg = load_config(args.config)
    run = Path(args.run)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, ckpt = load_checkpoint(run / "best.pt", device)
    cfg["label_map"] = {**cfg["label_map"], **ckpt["binary_map"]}
    split_col = f"split_{ckpt.get('split', 'group')}"
    report: dict = {"run": run.name}

    if args.ood:
        ood = build_catalog(args.ood, cfg["label_map"], cfg.get("category_aliases"))
        unknown = set(ood["category"]) - set(ckpt["classes"])
        if unknown:
            # objects the model was never trained on (e.g. "other": pen, phone, hand): there is no correct
            # class, so report how the alert layer would treat them (unsafe = would raise danger near mouth)
            report["unknown_objects"] = _unknown_report(model, ckpt, ood[~ood["category"].isin(ckpt["classes"])],
                                                        args.ood, device, cfg)
            for c, r in report["unknown_objects"].items():
                print(f"[ood] '{c}' (not a trained class, n={r['n']}): called unsafe {r['unsafe']:.1%}, "
                      f"uncertain {r['uncertain']:.1%}, safe {r['safe']:.1%}; top predictions {r['top_predicted']}")
            ood = ood[ood["category"].isin(ckpt["classes"])]
        # condition = first folder level that is not the class name (e.g. low_light, dim, bright)
        ood["condition"] = [next((p for p in Path(x).parts[:-1] if normalize_name(p) not in cfg["label_map"]), "all")
                            for x in ood["path"]]
        report["ood"] = _eval_df(model, ckpt, ood, args.ood, device, cfg, plot_prefix=run / "ood", title="OOD")
        report["ood_by_condition"] = {c: _eval_df(model, ckpt, g, args.ood, device, cfg)
                                      for c, g in ood.groupby("condition")} if ood["condition"].nunique() > 1 else {}
        test = pd.read_csv(args.manifest or cfg["paths"]["splits"])
        test = test[test[split_col] == "test"]
        ind = _eval_df(model, ckpt, test, source_roots(cfg), device, cfg)
        report["in_domain_test"] = ind
        report["ood_drop"] = {"accuracy": ind["accuracy"] - report["ood"]["accuracy"],
                              "unsafe_recall": ind["binary"]["unsafe_recall"] - report["ood"]["binary"]["unsafe_recall"]}
        print(f"in-domain acc {ind['accuracy']:.4f} -> OOD acc {report['ood']['accuracy']:.4f}; "
              f"unsafe recall {ind['binary']['unsafe_recall']:.4f} -> {report['ood']['binary']['unsafe_recall']:.4f}; "
              f"ECE(cal) {ind['ece_calibrated']:.4f} -> {report['ood']['ece_calibrated']:.4f}")

    if args.corruptions:
        test = pd.read_csv(args.manifest or cfg["paths"]["splits"])
        test = test[test[split_col] == "test"]
        curves = {}
        for kind in KINDS:
            curves[kind] = []
            for sev in range(0, 6):
                m = _eval_df(model, ckpt, test, source_roots(cfg), device, cfg, (kind, sev))
                curves[kind].append({"severity": sev, "accuracy": m["accuracy"],
                                     "unsafe_recall": m["binary"]["unsafe_recall"], "ece": m["ece_calibrated"]})
                print(f"{kind} s{sev}: acc {m['accuracy']:.4f} unsafe_recall {m['binary']['unsafe_recall']:.4f}")
        report["corruption_curves"] = curves
        _plot_curves(curves, run / "degradation.png", run.name)

    (run / "eval_report.json").write_text(json.dumps(report, indent=2))
    print(f"wrote {run / 'eval_report.json'}")


def _plot_curves(curves, path, title):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    for kind, pts in curves.items():
        s = [p["severity"] for p in pts]
        axes[0].plot(s, [p["accuracy"] for p in pts], marker="o", label=kind)
        axes[1].plot(s, [p["unsafe_recall"] for p in pts], marker="o", label=kind)
    axes[0].set_title("accuracy"); axes[1].set_title("unsafe recall")
    for a in axes:
        a.set_xlabel("severity"); a.set_ylim(0, 1.02); a.grid(alpha=0.3)
    axes[1].legend(fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/childsafe/classifier/gradcam.py
"""Grad-CAM (Selvaraju et al., ICCV 2017) overlays for qualitative inspection.

    python -m childsafe.classifier.gradcam --run outputs/runs/efficientnet_b0_group --n 40
    python -m childsafe.classifier.gradcam --run ... --images data/ood/knife   # any folder
"""
from __future__ import annotations

import argparse
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

from childsafe.classifier.calibration import softmax
from childsafe.classifier.models import gradcam_target_layer, load_checkpoint
from childsafe.config import load_config
from childsafe.data.catalog import IMG_EXT, source_roots
from childsafe.data.dataset import eval_transform, image_paths


class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        layer.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        # tensor hook instead of a module backward hook: timm uses in-place activations
        self.acts = out
        out.register_hook(lambda g: setattr(self, "grads", g))

    def __call__(self, x: torch.Tensor, target: int | None = None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        target = int(logits.argmax(1)) if target is None else target
        logits[0, target].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), logits.detach().cpu().numpy()[0], target


def overlay(rgb: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1]
    return (rgb * (1 - alpha) + heat * alpha).astype(np.uint8)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True)
    ap.add_argument("--images", default=None, help="folder of images; default: random test-split images")
    ap.add_argument("--n", type=int, default=30)
    ap.add_argument("--config", default=None)
    args = ap.parse_args(argv)

    cfg = load_config(args.config)
    run = Path(args.run)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, ckpt = load_checkpoint(run / "best.pt", device)
    cam = GradCAM(model, gradcam_target_layer(model, ckpt["arch"]))
    size = ckpt["img_size"]
    tf = eval_transform(size)
    show = eval_transform(size).transforms[:2]  # resize + crop only, for display

    if args.images:
        paths = sorted(p for p in Path(args.images).rglob("*") if p.suffix.lower() in IMG_EXT)[: args.n]
        truth = [""] * len(paths)
    else:
        df = pd.read_csv(cfg["paths"]["splits"])
        df = df[df[f"split_{ckpt.get('split', 'group')}"] == "test"].sample(frac=1, random_state=0).head(args.n)
        paths = image_paths(df, source_roots(cfg))
        truth = list(df["category"])

    out = run / "gradcam"
    out.mkdir(exist_ok=True)
    for k, (p, t) in enumerate(zip(paths, truth)):
        img = Image.open(p).convert("RGB")
        x = tf(img).unsqueeze(0).to(device).requires_grad_(True)
        heat, logits, target = cam(x)
        prob = softmax(logits[None], ckpt["temperature"])[0]
        disp = np.asarray(show[1](show[0](img)))
        vis = overlay(disp, heat)
        label = f"pred {ckpt['classes'][target]} {prob[target]:.2f}" + (f" | true {t}" if t else "")
        vis = cv2.putText(vis.copy(), label, (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
        Image.fromarray(vis).save(out / f"{k:03d}_{Path(p).stem}.jpg")
    print(f"wrote {len(paths)} overlays to {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/inspect_dataset.py
"""Task 2 of the handoff: record folder structure, class counts, image sizes and signs of
pre-applied augmentation. Writes docs/dataset_report.md.

    python scripts/inspect_dataset.py
"""
import argparse
import re
from collections import Counter
from pathlib import Path

from PIL import Image

from childsafe.config import load_config
from childsafe.data.catalog import IMG_EXT, build_catalog

AUG_HINT = re.compile(r"(aug|augment|_rot|_flip|copy|\(\d+\)|_\d+_\d+)", re.I)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=None)
    cfg = load_config(ap.parse_args().config)
    root = Path(cfg["paths"]["raw_data"])
    lines = [f"# ChildSUn dataset report\n", f"Root: `{root}`\n", "## Folder tree (image counts)\n", "```"]
    counts = Counter()
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXT:
            counts[p.parent.relative_to(root).as_posix()] += 1
    for k in sorted(counts):
        lines.append(f"{k}: {counts[k]}")
    lines.append("```\n")
    other = Counter(p.suffix.lower() for p in root.rglob("*") if p.is_file() and p.suffix.lower() not in IMG_EXT)
    if other:
        lines.append(f"Non-image files by extension: {dict(other)}\n")

    df = build_catalog(root, cfg["label_map"], cfg.get("category_aliases"))
    lines.append(f"## Catalog\n\nLabelled images: **{len(df)}**\n")
    lines.append("| category | binary | count |\n|---|---|---|")
    for (c, b), n in df.groupby(["category", "label"]).size().items():
        lines.append(f"| {c} | {b} | {n} |")
    lines.append(f"\nBinary totals: {df['label'].value_counts().to_dict()}\n")
    if df["source_split"].str.len().gt(0).any():
        lines.append(f"Dataset ships its own split folders: {df['source_split'].value_counts().to_dict()} "
                     "(not used; see A1).\n")

    sizes = Counter()
    for p in df["path"].sample(min(500, len(df)), random_state=0):
        try:
            with Image.open(root / p) as im:
                sizes[im.size] += 1
        except Exception:
            sizes["unreadable"] += 1
    lines.append(f"Image sizes (sample of {sum(sizes.values())}): {dict(sizes.most_common(8))}\n")
    hinted = df["path"].map(lambda s: bool(AUG_HINT.search(Path(s).stem)))
    lines.append(f"File names matching augmentation-like patterns: {int(hinted.sum())} "
                 f"(e.g. {list(df.loc[hinted, 'path'].head(5))})\n")
    lines.append("\n## Still to check by hand\n\n- License (see dataset-metadata.json from the download script)\n"
                 "- Whether augmented copies are included (compare with the ~1,358 raw / ~5,350 total in the paper abstract; "
                 "see near-duplicate groups from scripts/make_splits.py)\n- Safe/unsafe grouping of each category vs the paper\n")
    out = Path(cfg["paths"]["docs"]) / "dataset_report.md"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text("\n".join(lines), encoding="utf-8")
    print("\n".join(lines))
    print(f"\nwrote {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/make_splits.py
"""Task 3 (A1): perceptual hashes -> near-duplicate groups -> group split and random split.

Writes data/splits/splits.csv (path, category, label, group, split_group, split_random),
docs/split_report.md and outputs/dedup/pairs_montage_*.jpg for manual threshold checks.

    python scripts/make_splits.py
    python scripts/make_splits.py --phash 8 --dhash 10   # override thresholds
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

from childsafe.config import ROOT, load_config
from childsafe.data.catalog import build_full_catalog, source_roots
from childsafe.data.dataset import image_paths
from childsafe.data.dedup import (compute_hashes, cross_split_leakage, find_duplicate_pairs,
                                  union_find_groups)
from childsafe.data.split import group_split, random_split


def montage(full_paths, pairs, path, n=24, thumb=112):
    rng = np.random.default_rng(0)
    sel = pairs[rng.permutation(len(pairs))[:n]] if len(pairs) else pairs
    canvas = Image.new("RGB", (thumb * 4, thumb * ((len(sel) + 1) // 2)), "white")
    for k, (i, j, dp, dd) in enumerate(sel):
        for m, idx in enumerate((i, j)):
            im = Image.open(full_paths[idx]).convert("RGB").resize((thumb, thumb))
            canvas.paste(im, ((k % 2) * 2 * thumb + m * thumb, (k // 2) * thumb))
        ImageDraw.Draw(canvas).text(((k % 2) * 2 * thumb + 2, (k // 2) * thumb + 2), f"p{dp} d{dd}", fill="yellow")
    canvas.save(path)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--phash", type=int)
    ap.add_argument("--dhash", type=int)
    ap.add_argument("--rehash", action="store_true")
    ap.add_argument("--config", default=None)
    args = ap.parse_args()
    cfg = load_config(args.config)
    dc, sc = cfg["dedup"], cfg["split"]
    tp, td = args.phash or dc["phash_threshold"], args.dhash or dc["dhash_threshold"]
    roots = source_roots(cfg)

    df = build_full_catalog(cfg)
    full_paths = image_paths(df, roots)
    keys = [f"{s}/{p}" for s, p in zip(df.source, df.path)]
    hash_file = Path(cfg["paths"]["hashes"])
    hash_file.parent.mkdir(parents=True, exist_ok=True)
    if hash_file.exists() and not args.rehash and list(np.load(hash_file, allow_pickle=True)["paths"]) == keys:
        z = np.load(hash_file, allow_pickle=True)
        ph, dh = z["phash"], z["dhash"]
    else:
        ph, dh = compute_hashes(full_paths, dc["hash_size"], dc["dihedral"])
        np.savez(hash_file, phash=ph, dhash=dh, paths=np.array(keys))

    report = ["# Split report (A1)\n", f"Images: {len(df)}. Dihedral hashing: {dc['dihedral']}.\n",
              "## Threshold sensitivity\n", "| pHash thr | dHash thr | dup pairs | groups | largest group | images in groups >1 |",
              "|---|---|---|---|---|---|"]
    for t in sorted({4, 6, 8, 10, 12, 14, tp}):
        pr = find_duplicate_pairs(ph, dh, t, t + (td - tp))
        g = union_find_groups(len(df), pr)
        sizes = np.bincount(g)
        report.append(f"| {t} | {t + td - tp} | {len(pr)} | {len(sizes)} | {sizes.max()} | {int(sizes[sizes > 1].sum())} |")

    pairs = find_duplicate_pairs(ph, dh, tp, td)
    # Filename evidence: augmented copies of one source photo (ChildSUn: <class>_<index>_0_<rand>.jpg).
    src_codes = df["source_id"].astype("category").cat.codes.to_numpy()
    src_pairs = np.array([(i, j, -1, -1) for idx in pd.Series(range(len(df))).groupby(src_codes).groups.values()
                          for i, j in zip(list(idx)[:-1], list(idx)[1:])], dtype=np.int64).reshape(-1, 4)
    hash_groups = union_find_groups(len(df), pairs)
    groups = union_find_groups(len(df), np.concatenate([pairs, src_pairs]))
    sizes = np.bincount(groups)
    n_src = df["source_id"].nunique()
    # how well does pHash alone recover the filename groups? (validates A1 for datasets without such names)
    multi = df.groupby("source_id").filter(lambda g: len(g) > 1)
    caught = sum(len(set(hash_groups[g.index])) == 1 for _, g in multi.groupby("source_id"))
    n_multi = multi["source_id"].nunique()
    report += [f"\n## Source photos from filenames\n",
               f"Distinct source photos: {n_src} (paper abstract: ~1,358 raw). "
               f"Sources with several augmented copies: {n_multi}.",
               f"pHash/dHash alone puts all copies of a source in one group for {caught}/{n_multi} sources "
               f"({caught / max(n_multi, 1):.1%}). Final groups = union of filename groups and hash groups.\n"]
    if sizes.max() > 0.05 * len(df):
        print(f"WARNING: largest group has {sizes.max()} images (>5%). Threshold may be too loose (chaining).")
    df["group"] = groups
    y = df["category"].to_numpy()
    base = sc.get("base_splits")
    if base and (df.source != "childsun").any():
        # Extension data: keep every ChildSUn image in the split of the canonical ChildSUn split, so test
        # results stay comparable; only split the extra images (grouped), and let an extra image that is a
        # near-duplicate of a ChildSUn image follow that image. Bridges between ChildSUn splits are dropped.
        b = pd.read_csv(ROOT / base).set_index("path")
        cs = (df.source == "childsun").to_numpy()
        for col in ("split_group", "split_random"):
            df[col] = None
            df.loc[cs, col] = b.loc[df.path[cs], col].to_numpy()
        fixed = df[cs].groupby("group")["split_group"].agg(lambda s: s.iloc[0] if s.nunique() == 1 else "conflict")
        extra = ~cs
        follow = extra & df.group.isin(fixed.index).to_numpy()
        df.loc[follow, "split_group"] = df.loc[follow, "group"].map(fixed).to_numpy()
        free = extra & ~follow
        df.loc[free, "split_group"] = group_split(y[free], groups[free], sc["test_frac"], sc["val_frac"], sc["seed"])
        df.loc[extra, "split_random"] = random_split(y[extra], sc["test_frac"], sc["val_frac"], sc["seed"])
        df.loc[df.split_group == "conflict", "split_group"] = "excluded"   # kept as rows so hash arrays stay aligned
        n_conf = int((df.split_group == "excluded").sum())
        report.append(f"\nChildSUn rows keep the split of `{base}`. Extra images that are near-duplicates of a ChildSUn "
                      f"image follow it: {int(follow.sum())}; dropped as bridges between ChildSUn splits: {n_conf}.\n")
    else:
        df["split_group"] = group_split(y, groups, sc["test_frac"], sc["val_frac"], sc["seed"])
        df["split_random"] = random_split(y, sc["test_frac"], sc["val_frac"], sc["seed"])
    out_csv = Path(cfg["paths"]["splits"])
    df.to_csv(out_csv, index=False)
    import hashlib
    cs_rows = df[df.get("source", "childsun") == "childsun"]
    checksum = hashlib.md5("".join(f"{p}:{g}" for p, g in sorted(zip(cs_rows.path, cs_rows.split_group))).encode()).hexdigest()[:12]
    print(f"ChildSUn group split checksum: {checksum}  (identical on every machine = identical split)")

    report += [f"\n## Chosen thresholds: pHash <= {tp}, dHash <= {td}\n",
               f"Near-duplicate pairs: {len(pairs)}; groups: {len(sizes)}; largest: {sizes.max()}; "
               f"images sharing a group: {int(sizes[sizes > 1].sum())} ({sizes[sizes > 1].sum() / len(df):.1%}).\n",
               "## Split sizes\n", "| split | group split | random split |", "|---|---|---|"]
    for s in ("train", "val", "test"):
        report.append(f"| {s} | {(df.split_group == s).sum()} | {(df.split_random == s).sum()} |")
    report += ["\n## Cross-split near-duplicates (share of test images with a near-duplicate in train)\n",
               f"- random split: {cross_split_leakage(ph, dh, df.split_random.to_numpy(), tp, td):.1%}",
               f"- group split: {cross_split_leakage(ph, dh, df.split_group.to_numpy(), tp, td):.1%} (should be 0)\n",
               "Share of test images whose source photo (filename) also has a copy in train:\n",
               *[f"- {name} split: {df[df[col] == 'test'].source_id.isin(set(df[df[col] == 'train'].source_id)).mean():.1%}"
                 for name, col in (("random", "split_random"), ("group", "split_group"))],
               "",
               "Class balance of test split (group):\n", "```", df[df.split_group == 'test'].category.value_counts().to_string(), "```\n",
               "Check `outputs/dedup/pairs_montage_*.jpg` to confirm the pairs really are duplicates / augmentations."]
    mdir = Path(cfg["paths"]["outputs"]) / "dedup"
    mdir.mkdir(parents=True, exist_ok=True)
    montage(full_paths, pairs, mdir / f"pairs_montage_p{tp}_d{td}.jpg")
    # borderline pairs just above threshold: are we missing real duplicates?
    near = find_duplicate_pairs(ph, dh, tp + 4, td + 4)
    border = near[(near[:, 2] > tp) | (near[:, 3] > td)]
    montage(full_paths, border, mdir / f"pairs_montage_borderline.jpg")

    Path(cfg["paths"]["docs"]).mkdir(parents=True, exist_ok=True)
    (Path(cfg["paths"]["docs"]) / "split_report.md").write_text("\n".join(report), encoding="utf-8")
    print("\n".join(report))
    print(f"\nwrote {out_csv}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/train_all.py
"""Tasks 4-6: train every model on the group split and the random split, then tabulate.

    python scripts/train_all.py                  # all models x {group, random}
    python scripts/train_all.py --summary-only   # just rebuild the table from existing runs
"""
import argparse
import json
from pathlib import Path

from childsafe.classifier import train
from childsafe.config import load_config


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--summary-only", action="store_true")
    ap.add_argument("--models", nargs="*")
    ap.add_argument("--splits", nargs="*", default=["group", "random"])
    ap.add_argument("--robust", action="store_true")
    ap.add_argument("--config", default=None)
    args = ap.parse_args()
    cfg = load_config(args.config)
    models = args.models or cfg["train"]["models"]
    runs = Path(cfg["paths"]["outputs"]) / "runs"

    if not args.summary_only:
        for arch in models:
            for split in args.splits:
                train.main(["--arch", arch, "--split", split] + (["--robust"] if args.robust else [])
                           + (["--config", args.config] if args.config else []))

    rows = ["| model | split | acc | macro-F1 | unsafe recall | unsafe precision | ECE raw | ECE cal | T |",
            "|---|---|---|---|---|---|---|---|---|"]
    for f in sorted(runs.glob("*/metrics.json")):
        m = json.loads(f.read_text())
        t = m["test"]
        rows.append(f"| {m['arch']}{' +A5' if m['robust'] else ''} | {m['split']} | {t['accuracy']:.4f} | "
                    f"{t['macro_f1']:.4f} | {t['binary']['unsafe_recall']:.4f} | {t['binary']['unsafe_precision']:.4f} | "
                    f"{t['ece_raw']:.4f} | {t['ece_calibrated']:.4f} | {t['temperature']:.2f} |")
    text = ("# Module A results (test split)\n\nGenerated by scripts/train_all.py. The random-split rows can be inflated by "
            "near-duplicates shared between train and test; the gap to the group-split rows estimates that leakage.\n\n"
            + "\n".join(rows) + "\n")
    Path(cfg["paths"]["docs"]).mkdir(parents=True, exist_ok=True)
    (Path(cfg["paths"]["docs"]) / "results_moduleA.md").write_text(text, encoding="utf-8")
    print(text)


if __name__ == "__main__":
    main()

## 3. Locate the dataset and write the Kaggle config

In [ ]:
imgs = glob.glob("/kaggle/input/**/*.jpg", recursive=True)
assert imgs, "No images found: add the ChildSUn dataset with 'Add Input'."
# the folder holding most images (ChildSUn ships flat: class is the filename prefix)
from collections import Counter
DATA = Counter(os.path.dirname(p) for p in imgs).most_common(1)[0][0]
print(len(imgs), "images; using", DATA)

FAST = False  # True: 8 epochs per run for a quick check; False: config default (25, early stopping)
cfg = f'''_base_: default.yaml
paths:
  raw_data: {DATA}
train:
  num_workers: 4
  batch_size: 32
'''
if FAST:
    cfg += "  epochs: 8\n"
open("configs/kaggle.yaml", "w").write(cfg)
print(cfg)

## 4. Dataset report and leak-free split (A1)

In [ ]:
!python scripts/inspect_dataset.py --config configs/kaggle.yaml

In [ ]:
!python scripts/make_splits.py --config configs/kaggle.yaml 2>&1 | grep -v "it/s"

In [ ]:
from IPython.display import Image as Show, display
for p in sorted(glob.glob("outputs/dedup/*.jpg")):
    print(p); display(Show(p, width=420))

## 5. Train 3 models × {group, random} split

Each run: train with early stopping on val macro-F1 → fit temperature on val → evaluate on test.
About 1–2 h in total on a T4 (VGG19 is the slowest). To run a subset, edit `MODELS` / `SPLITS`.

In [ ]:
MODELS = "efficientnet_b0 mobilenetv3_large_100 vgg19_bn"
SPLITS = "group random"
!python scripts/train_all.py --config configs/kaggle.yaml --models {MODELS} --splits {SPLITS} 2>&1 | grep -vE "UserWarning|sched.step"

In [ ]:
print(open("docs/results_moduleA.md").read())

## 6. Calibration plots, corruption curves (A5) and Grad-CAM — group-split runs

In [ ]:
for run in sorted(glob.glob("outputs/runs/*_group")):
    print("=" * 20, run)
    !python -m childsafe.classifier.evaluate --run {run} --config configs/kaggle.yaml --corruptions 2>&1 | grep -E "s0|s5|wrote"
    !python -m childsafe.classifier.gradcam --run {run} --config configs/kaggle.yaml --n 24

In [ ]:
for run in sorted(glob.glob("outputs/runs/*_group")):
    for name in ["test_reliability_raw.png", "test_reliability_calibrated.png", "degradation.png"]:
        p = f"{run}/{name}"
        if os.path.exists(p):
            print(p); display(Show(p, width=480))

In [ ]:
for p in sorted(glob.glob("outputs/runs/efficientnet_b0_group/gradcam/*.jpg"))[:12]:
    display(Show(p, width=224))

## 7. Optional: A5 robustness augmentation, then out-of-domain test (A2)

For A2, upload your own webcam photos as a Kaggle dataset with one folder per class
(e.g. `knife/`, `toy/`, optionally `low_light/knife/...`) and set `OOD` to its path.

In [ ]:
RUN_ROBUST = False
if RUN_ROBUST:
    !python -m childsafe.classifier.train --arch efficientnet_b0 --split group --robust --config configs/kaggle.yaml
    !python -m childsafe.classifier.evaluate --run outputs/runs/efficientnet_b0_group_robust --config configs/kaggle.yaml --corruptions

OOD = ""   # e.g. "/kaggle/input/my-webcam-photos"
if OOD:
    for run in sorted(glob.glob("outputs/runs/*_group*")):
        !python -m childsafe.classifier.evaluate --run {run} --config configs/kaggle.yaml --ood {OOD}

## 8. Download results

The zip includes checkpoints (`best.pt`) so you can run the webcam demo locally with them.

In [ ]:
!cd /kaggle/working && zip -qr childsafe_results.zip childsafe/outputs childsafe/docs childsafe/data/splits -x "*hashes.npz" "*_random/best.pt"  # random-split checkpoints skipped (VGG19 is ~550 MB)
from IPython.display import FileLink
FileLink("childsafe_results.zip")